In [ ]:
!git clone https://github.com/t-ravnushkin/generalized-toric-gpu-search.git
import sys, os
sys.path.insert(0, "/kaggle/working/generalized-toric-gpu-search")
os.chdir("/kaggle/working")

# GF(8) Toric Code Champion Search — Kaggle (NVIDIA GPU)

OpenCL runs natively on NVIDIA via the CUDA toolkit ICD — the kernel is unchanged.

In [ ]:
!pip install pyopencl -q
os.environ.setdefault("PYOPENCL_CTX", "0")

In [ ]:
import pyopencl as cl
for p in cl.get_platforms():
    for d in p.get_devices():
        kind = 'GPU' if d.type == cl.device_type.GPU else 'CPU'
        print(f"{p.name}  |  {d.name}  |  {kind}")

In [ ]:
from precompute import init_opencl_all, bounding_cube_points
from kernel import DistanceOracle
from champion_search import find_latest_results, load_results, check_set, global_batched_bfs, idx
from pathlib import Path
from datetime import datetime

gpu_contexts, _ = init_opencl_all()
oracles = [DistanceOracle(ctx, queue, M_buf) for ctx, queue, M_buf in gpu_contexts]
lattice = bounding_cube_points()
print(f"\n{len(oracles)} GPU(s) ready.")

# Adaptive batch size based on the first (or only) GPU.
# Each GPU gets batch_size / n_gpus sets — total work stays constant.
sm_count = oracles[0].ctx.devices[0].max_compute_units
ADAPTIVE_BATCH = sm_count * 1024 * 8 * len(oracles)   # scale with GPU count
print(f"Batch size: {ADAPTIVE_BATCH:,}  ({sm_count} SMs × 8 waves × {len(oracles)} GPU(s))")

## Codetables.de — best known bounds for [49, k] codes over GF(8)

These are the targets our toric codes are racing against.
A result matching or beating the table entry would be a new record.

In [ ]:
from codetables import bounds_for_n
import pandas as pd

targets = bounds_for_n(49)   # fetched once, cached in .gf8_bounds_cache.txt

df_bounds = pd.DataFrame(
    [(k, d) for k, d in sorted(targets.items()) if k <= 15],
    columns=["k", "best_known_d"]
)
display(df_bounds)

In [ ]:
results_file = find_latest_results()
if results_file is None:
    results_file = Path(f"champions_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json")
    print(f"Starting fresh: {results_file}")
else:
    print(f"Resuming from: {results_file}  ({len(load_results(results_file))} records)")

## Part A — Structured geometric candidates

In [ ]:
already_done = (
    {rec["name"] for rec in load_results(results_file) if "name" in rec}
    if results_file.exists() else set()
)

circle_8 = sorted([idx(0,1), idx(0,6), idx(1,0), idx(2,2), idx(2,5), idx(5,2), idx(5,5), idx(6,0)])
check_set("circle_conic_k8", circle_8, oracles[0], lattice, results_file, already_done=already_done)

parabola_7 = sorted([idx(t, (t*t) % 7) for t in range(7)])
check_set("parabola_conic_k7", parabola_7, oracles[0], lattice, results_file, already_done=already_done)

## Part B — Global BFS guided by codetables.de bounds

`targets` is the per-k dict from the table, so at each level k the BFS only
keeps sets whose minimum distance meets the best known general-code bound.
Reduce `targets[k]` by some margin if you want a wider net.

In [ ]:
import time

# margin = 5
# targets = {k: max(d - margin, 1) for k, d in targets.items()}

t0 = time.perf_counter()
global_batched_bfs(
    target_distance=targets,
    oracles=oracles,          # list — work is split across all GPUs
    lattice=lattice,
    results_file=results_file,
    max_k=10,
    resume=True,
    batch_size=ADAPTIVE_BATCH,
)
print(f"\nTotal BFS time: {time.perf_counter() - t0:.1f}s")

## Results vs. best known bounds

In [ ]:
records = [r for r in load_results(results_file) if r.get("type") != "level_complete"]
df = pd.DataFrame(records)[["name", "k", "min_distance", "indices"]]
df["best_known_d"] = df["k"].map(targets)
df["gap"] = df["best_known_d"] - df["min_distance"]
df["new_record"] = df["gap"] < 0
df.sort_values(["k", "min_distance"], ascending=[True, False], inplace=True)
df.reset_index(drop=True, inplace=True)
display(df)